# Conversión Masiva de PDFs a Markdown con MinerU — ALEsys

Notebook actualizado para procesar todos los PDFs pendientes del directorio `pdf_pendientes/`.
Diseñado para Google Colab con GPU T4.

**PDFs pendientes:**
- Landau (10 volúmenes, ~240MB)
- Goldstein 3rd ed (15 capítulos, ~200MB)
- QuantumMechanicsScript.pdf (59MB)
- Cohen-Tannoudji Vol 1 (38MB)
- Kouh (35MB)
- Greiner intro (21MB)
- Greiner special chapters (11MB)
- Métodos matemáticos (7 PDFs, 118MB)
- Ondas y óptica (4 PDFs, 198MB)
- Física térmica (2 PDFs)
- Grover 1996 (1 PDF)
- Chaos – Physics Open Lab (1 PDF)
- Davide Iannuzzi (1 PDF)
- Jose Latorre (1 PDF)
- Y más en `2_mas_libros/` (5 libros adicionales)

**Total estimado: ~431 PDFs, ~1.5GB**


## 1. Montar Google Drive y establecer rutas

In [ ]:
from google.colab import drive
from pathlib import Path

print("Montando Google Drive...")
drive.mount('/content/drive', force_remount=True)

# Rutas base — ajustar según tu estructura
base_drive = Path("/content/drive/MyDrive/db_p")
base_procesados = Path("/content/drive/MyDrive/db_p_procesado")
pendientes_dir = base_procesados / "pdf_pendientes"

print(f"base_drive: {base_drive} (existe: {base_drive.exists()})")
print(f"base_procesados: {base_procesados} (existe: {base_procesados.exists()})")
print(f"pendientes_dir: {pendientes_dir} (existe: {pendientes_dir.exists()})")


## 2. Instalar MinerU y dependencias (versiones actualizadas 2026)

In [ ]:
import os, sys, shutil, subprocess

print("Limpiando e instalando dependencias...")

# Desinstalar versiones anteriores conflictivas
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "mineru", "magic-pdf", "transformers"], check=False)

# Instalar transformers compatible (version estable)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "--quiet", "transformers>=4.40.0,<5.0.0"], check=False)

# Instalar MinerU con soporte core (incluye pipeline backend)
result = subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "--quiet", "mineru[core]"], check=False)
if result.returncode == 0:
    print("MinerU instalado correctamente")
else:
    print("Posible problema instalando MinerU")

# Dependencias auxiliares
for pkg in ["tqdm", "langdetect", "pdfminer.six", "pandas", "pypdf"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=False)

# Configurar entorno
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Entorno listo")


## 3. Verificar GPU disponible

In [ ]:
import subprocess, os

print("Verificando GPU...")
try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    if result.returncode == 0:
        print(result.stdout[:500])
    else:
        print("No se detecto GPU (nvidia-smi fallo)")
except FileNotFoundError:
    print("nvidia-smi no encontrado - ejecutando en CPU")
except Exception as e:
    print(f"Error: {e}")

print(f"\nCUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', 'no establecida')}")


## 4. Verificar MinerU

In [ ]:
import shutil, subprocess, sys, time, urllib.request

print("Verificando mineru...")
mineru_path = shutil.which("mineru")
if mineru_path:
    print(f"mineru encontrado en: {mineru_path}")
else:
    print("mineru no en PATH, se usara 'python -m mineru'")

# Verificar mineru-api
api_exec = shutil.which("mineru-api")
if api_exec:
    print(f"mineru-api encontrado en: {api_exec}")
    try:
        help_proc = subprocess.run([api_exec, "--help"], capture_output=True, text=True, timeout=20)
        print(help_proc.stdout[:800])
    except Exception as e:
        print(f"Error: {e}")
else:
    print("mineru-api no encontrado")


## 5. Listar PDFs pendientes de procesar

In [ ]:
from pathlib import Path

pdf_files = sorted(base_drive.rglob("*.pdf"))
print(f"Total PDFs en base_drive: {len(pdf_files)}")

# También listar los que están en pdf_pendientes
pendientes_pdfs = []
if pendientes_dir.exists():
    pendientes_pdfs = sorted(pendientes_dir.rglob("*.pdf"))
    print(f"PDFs en pdf_pendientes: {len(pendientes_pdfs)}")

# Combinar y deduplicar
all_pdfs = list(set(pdf_files + pendientes_pdfs))
print(f"Total unicos: {len(all_pdfs)}")

# Filtrar ya procesados
candidates = []
for pdf in all_pdfs:
    try:
        rel = pdf.relative_to(base_drive) if base_drive in pdf.parents or pdf == base_drive else None
    except ValueError:
        rel = None
    dest_dir = base_procesados / (rel.parent if rel else Path(".")) / pdf.stem
    md_files = list(dest_dir.rglob("*.md")) if dest_dir.exists() else []
    if md_files:
        continue
    candidates.append((pdf, dest_dir))

print(f"\nPDFs pendientes de procesar: {len(candidates)}")
for pdf, dest in candidates[:30]:
    print(f"  {pdf.name}")
if len(candidates) > 30:
    print(f"  ... y {len(candidates) - 30} mas")


## 6. Configurar concurrencia y timeout

In [ ]:
import time
from tqdm.notebook import tqdm

MAX_WORKERS = 1  # 1 para GPU T4 (evitar OOM)
TIMEOUT_SECONDS = 20 * 3600  # 20 horas por libro
BACKEND = "pipeline"  # pipeline = CPU+GPU, vlm-auto-engine = VLM (mas lento, mejor calidad)
AUTO_DETECT_LANG = True
DEFAULT_LANG = None

print(f"Configuracion:")
print(f"  MAX_WORKERS: {MAX_WORKERS}")
print(f"  TIMEOUT: {TIMEOUT_SECONDS // 3600} horas")
print(f"  BACKEND: {BACKEND}")
print(f"  AUTO_DETECT_LANG: {AUTO_DETECT_LANG}")


## 7. Deteccion de idioma

In [ ]:
import subprocess, sys
from pathlib import Path

try:
    from langdetect import detect, detect_langs
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "langdetect"], check=False)
    from langdetect import detect, detect_langs

try:
    from pdfminer.high_level import extract_text
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pdfminer.six"], check=False)
    from pdfminer.high_level import extract_text

def extract_pdf_text_sample(pdf_path, max_pages=3, chars_per_page=500):
    try:
        text = extract_text(str(pdf_path), maxpages=max_pages)
        return text[:chars_per_page * max_pages] if text else ""
    except Exception as e:
        return ""

def detect_language(text):
    if not text or len(text) < 50:
        return ("en", 0.5)
    try:
        lang_code = detect(text)
        probs = detect_langs(text)
        confidence = next((p.prob for p in probs if p.lang == lang_code), 0.5)
        return (lang_code, confidence)
    except Exception:
        return ("en", 0.3)

def map_language_to_mineru(lang_code):
    mapping = {
        "en": "en", "es": "en", "fr": "en", "de": "en",
        "pt": "en", "it": "en", "nl": "en", "pl": "en",
        "ru": "cyrillic", "uk": "cyrillic", "ar": "arabic",
        "el": "el", "ja": "japan", "ko": "korean",
        "zh-cn": "ch", "zh-tw": "chinese_cht", "hi": "devanagari",
        "ta": "ta", "te": "te", "ka": "ka", "th": "th",
    }
    simplified = lang_code.lower().split("-")[0]
    return mapping.get(simplified, "en")

def detect_pdf_language(pdf_path):
    text = extract_pdf_text_sample(pdf_path)
    lang_code, confidence = detect_language(text)
    mineru_lang = map_language_to_mineru(lang_code)
    return (lang_code, mineru_lang, confidence)

print("Funciones de deteccion de idioma cargadas")


## 8. Funcion de conversion con MinerU

In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path

log_root = base_procesados / "_logs"
log_root.mkdir(parents=True, exist_ok=True)

SERVICE_HOST = "127.0.0.1"
SERVICE_PORT = 7860
SERVICE_URL = f"http://{SERVICE_HOST}:{SERVICE_PORT}"
SERVICE_PROCESS = None

def is_port_open(host, port):
    import socket
    try:
        with socket.create_connection((host, port), timeout=2):
            return True
    except (OSError, ConnectionRefusedError):
        return False

def wait_for_service_ready(url, timeout=120):
    import urllib.request
    start = time.time()
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(f"{url}/openapi.json", timeout=5) as resp:
                if resp.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(2)
    return False

def start_mineru_service(host=SERVICE_HOST, port=SERVICE_PORT):
    import subprocess
    api_exec = shutil.which("mineru-api")
    if api_exec is None:
        api_exec = "mineru-api"
    cmd = [api_exec, "--host", host, "--port", str(port)]
    print(f"Iniciando mineru-api: {' '.join(cmd)}")
    proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
                           env=os.environ.copy())
    if wait_for_service_ready(SERVICE_URL, timeout=120):
        print("Servicio mineru-api listo")
    else:
        print("ADVERTENCIA: servicio no respondio, intentando de todas formas")
    return proc

def stop_mineru_service(proc):
    if proc is not None:
        try:
            proc.terminate()
            proc.wait(timeout=30)
            print("Servicio mineru-api detenido")
        except Exception as e:
            print(f"Error deteniendo servicio: {e}")

def get_mineru_client_command(api_url=SERVICE_URL):
    mineru_path = shutil.which("mineru")
    if mineru_path:
        return [mineru_path, "--api-url", api_url]
    return [sys.executable, "-m", "mineru", "--api-url", api_url]

def convert_pdf_with_mineru(pdf_path, output_dir, backend="pipeline", language=None,
                           auto_detect_lang=True, timeout=TIMEOUT_SECONDS, retries=1):
    output_dir.mkdir(parents=True, exist_ok=True)
    log_file = log_root / f"{pdf_path.stem}.log"

    detected_lang = language
    lang_confidence = 0.0
    if auto_detect_lang and (language is None or language == "auto"):
        try:
            lang_orig, lang_mineru, confidence = detect_pdf_language(pdf_path)
            detected_lang = lang_mineru
            lang_confidence = confidence
            with open(log_file, "a", encoding="utf-8") as log:
                log.write(f"Idioma detectado: {lang_orig} -> {lang_mineru} (confianza: {confidence:.1%})\n")
        except Exception as e:
            with open(log_file, "a", encoding="utf-8") as log:
                log.write(f"Error detectando idioma: {e}\n")
            detected_lang = "en"
            lang_confidence = 0.0

    if detected_lang is None:
        detected_lang = "en"

    command = [
        *get_mineru_client_command(SERVICE_URL),
        "-p", str(pdf_path),
        "-o", str(output_dir),
        "-m", "auto",
        "-b", backend,
        "-l", detected_lang,
        "--formula", "True",
        "--table", "True",
    ]

    env = os.environ.copy()
    env["OMP_NUM_THREADS"] = env.get("OMP_NUM_THREADS", "2")
    env["MKL_NUM_THREADS"] = env.get("MKL_NUM_THREADS", "2")
    env["OPENBLAS_NUM_THREADS"] = env.get("OPENBLAS_NUM_THREADS", "2")
    env["CUDA_VISIBLE_DEVICES"] = env.get("CUDA_VISIBLE_DEVICES", "0")

    for attempt in range(1, retries + 2):
        start = time.time()
        with open(log_file, "a", encoding="utf-8") as log:
            log.write(f"\n=== Intento {attempt} | {pdf_path.stem} ===\n")
            log.write(f"Backend: {backend} | Idioma: {detected_lang} (confianza: {lang_confidence:.1%})\n")
            log.write("Comando: " + " ".join(command) + "\n")
            log.flush()
            try:
                proc = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT,
                                     env=env, timeout=timeout, check=False, text=True)
                log.flush()
                elapsed = time.time() - start
                success = proc.returncode == 0
                log.write(f"\n[Returncode: {proc.returncode}]\n[Tiempo: {elapsed:.1f}s]\n")
                log.flush()

                if success:
                    log.write("Procesado exitosamente\n")
                    return {
                        "pdf": str(pdf_path),
                        "output_dir": str(output_dir),
                        "status": "ok",
                        "time_s": elapsed,
                        "attempt": attempt,
                        "language": detected_lang,
                        "lang_confidence": lang_confidence,
                        "log": str(log_file),
                    }

                if attempt <= retries:
                    log.write(f"Error en intento {attempt}, reintentando en 30s...\n")
                    log.flush()
                    time.sleep(30)
                    continue

                log.write(f"Error permanente tras {attempt} intentos\n")
                return {
                    "pdf": str(pdf_path),
                    "output_dir": str(output_dir),
                    "status": "error",
                    "time_s": elapsed,
                    "attempt": attempt,
                    "language": detected_lang,
                    "lang_confidence": lang_confidence,
                    "returncode": proc.returncode,
                    "log": str(log_file),
                }
            except subprocess.TimeoutExpired as exc:
                elapsed = time.time() - start
                log.write(f"\nTIMEOUT despues de {elapsed:.1f}s\n")
                if attempt <= retries:
                    log.write(f"Reintentando tras timeout (intento {attempt+1})...\n")
                    log.flush()
                    time.sleep(30)
                    continue
                log.write(f"Timeout permanente tras {attempt} intentos\n")
                return {
                    "pdf": str(pdf_path),
                    "output_dir": str(output_dir),
                    "status": "timeout",
                    "time_s": elapsed,
                    "attempt": attempt,
                    "language": detected_lang,
                    "lang_confidence": lang_confidence,
                    "log": str(log_file),
                }

def is_already_processed(pdf_path, output_dir):
    if not output_dir.exists():
        return False
    md_files = list(output_dir.rglob("*.md"))
    return len(md_files) > 0

print("Funcion convert_pdf_with_mineru definida")


## 9. Ejecutar procesamiento en lote

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import pandas as pd

# Recopilar PDFs pendientes
pdf_files = sorted(base_drive.rglob("*.pdf"))
# Tambien incluir los de pdf_pendientes si existe
if pendientes_dir.exists():
    pdf_files = sorted(set(pdf_files + list(pendientes_dir.rglob("*.pdf"))))

jobs = []
for pdf in pdf_files:
    try:
        rel = pdf.relative_to(base_drive)
        dest_dir = base_procesados / rel.parent / pdf.stem
    except ValueError:
        # PDF fuera de base_drive (ej. en pdf_pendientes)
        dest_dir = base_procesados / "pdf_pendientes_procesados" / pdf.stem
    if not is_already_processed(pdf, dest_dir):
        jobs.append((pdf, dest_dir))

print(f"Iniciando procesamiento de {len(jobs)} PDF(s)...")
results = []

service_proc = None
if len(jobs) > 0:
    try:
        service_proc = start_mineru_service()
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {
                executor.submit(convert_pdf_with_mineru, pdf, dest, BACKEND,
                               DEFAULT_LANG, AUTO_DETECT_LANG): (pdf, dest)
                for pdf, dest in jobs
            }
            for future in tqdm(as_completed(futures), total=len(futures), desc="Procesando"):
                result = future.result()
                results.append(result)
                pdf_name = Path(result["pdf"]).name
                lang_info = f" [{result.get('language', 'N/A')}]" if result.get("lang_confidence", 0) > 0 else ""
                print(f"  {pdf_name}: {result['status']} ({result['time_s']:.1f}s, intento {result['attempt']}){lang_info}")
    finally:
        stop_mineru_service(service_proc)
else:
    print("No hay trabajos pendientes")

# Resumen
if results:
    summary = pd.DataFrame(results)
    summary_file = base_procesados / "batch_conversion_summary.csv"
    summary.to_csv(summary_file, index=False)
    print(f"\nResumen guardado en: {summary_file}")
    ok_count = (summary["status"] == "ok").sum()
    error_count = (summary["status"] == "error").sum()
    timeout_count = (summary["status"] == "timeout").sum()
    print(f"Exitosos: {ok_count} | Errores: {error_count} | Timeouts: {timeout_count}")
    if len(results) > 0:
        print(f"Tiempo total: {summary['time_s'].sum()/3600:.1f} horas")


## 10. Revisar logs de problemas

In [ ]:
if 'summary' not in locals():
    print("La variable 'summary' no esta definida. Ejecuta la celda de procesamiento primero.")
else:
    logs_dir = base_procesados / "_logs"
    if logs_dir.exists():
        log_files = sorted(logs_dir.glob("*.log"))
        print(f"Logs disponibles: {len(log_files)}")
        error_results = summary[summary["status"] != "ok"]
        if len(error_results) > 0:
            print(f"Hay {len(error_results)} PDF(s) con problemas")
            for idx, row in error_results.tail(3).iterrows():
                log_path = Path(row["log"])
                if log_path.exists():
                    print(f"\n{'='*70}")
                    print(f"{row['pdf']} | Estado: {row['status']} | Intento: {row['attempt']}")
                    print(f"{'='*70}")
                    with open(log_path, "r") as f:
                        lines = f.read().splitlines()
                        print("\n".join(lines[-20:]))
        else:
            print("Todos los PDFs se procesaron correctamente")
    else:
        print("No hay directorio de logs")


## 11. Siguiente paso: Indexar en ALEsys

Una vez procesados los PDFs a Markdown, ejecutar en ALEsys:

```bash
curl -X POST http://localhost:3000/api/v1/ingestion/markdown \
  -H "Authorization: Bearer $TOKEN" \
  -d '{"dir": "/ruta/a/db_p_procesado", "dry_run": false}'
```

O usar la funcion de re-indexado para actualizar solo archivos modificados:

```bash
curl -X POST http://localhost:3000/api/v1/ingestion/markdown/reindex \
  -H "Authorization: Bearer $TOKEN" \
  -d '{"dir": "/ruta/a/db_p_procesado", "dry_run": false}'
```